# P2: Distributed Training on SageMaker

**Module:** 3 — MLOps
**Week:** 11

## What You'll Build
Scale training to 100K rows using SageMaker. Compare single-instance vs multi-instance training time and cost.

In [ ]:
import os
import boto3
import sagemaker
import pandas as pd
import numpy as np

In [ ]:
from src.generate_large_data import generate_large_churn_dataset
from sklearn.model_selection import train_test_split

df = generate_large_churn_dataset(100_000)
print(f"Dataset: {df.shape}, Churn rate: {df['churn'].mean():.2%}")
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['churn'], random_state=42)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

In [ ]:
import time
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# Preprocess
feature_cols = [c for c in df.columns if c not in ['customer_id', 'churn']]
X_train = train_df[feature_cols].fillna(train_df[feature_cols].median())
y_train = train_df['churn']
X_test = test_df[feature_cols].fillna(train_df[feature_cols].median())
y_test = test_df['churn']

# Time local training
start = time.time()
model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
model.fit(X_train, y_train)
local_time = time.time() - start

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Local training: {local_time:.1f}s, AUC={auc:.4f}")
print(f"Estimated SageMaker training (ml.m5.xlarge): ~{local_time*0.6:.0f}s")
print(f"Estimated SageMaker training (ml.m5.2xlarge x 2): ~{local_time*0.35:.0f}s")

---
## 2. Launch SageMaker Training Job

In [ ]:
# TODO: Upload train/test to S3
# import boto3, io
# s3 = boto3.client('s3')
# BUCKET = os.getenv('S3_BUCKET')
#
# for split, df_split in [('train', train_df), ('test', test_df)]:
#     buffer = io.StringIO()
#     df_split.to_csv(buffer, index=False)
#     s3.put_object(Bucket=BUCKET, Key=f'p2-distributed/{split}/data.csv', Body=buffer.getvalue())

# TODO: Configure and launch SKLearn Estimator
# from sagemaker.sklearn.estimator import SKLearn
# estimator = SKLearn(
#     entry_point='src/train.py',
#     framework_version='0.23-1',
#     instance_type='ml.m5.xlarge',
#     instance_count=1,
#     role=SAGEMAKER_ROLE,
#     use_spot_instances=True,
#     max_wait=600,
#     max_run=300,
# )
# estimator.fit({'train': f's3://{BUCKET}/p2-distributed/train/', 'test': f's3://{BUCKET}/p2-distributed/test/'})

In [ ]:
# TODO: Monitor training job
# sm = boto3.client('sagemaker')
# job = sm.describe_training_job(TrainingJobName=estimator.latest_training_job.name)
# print(f"Status: {job['TrainingJobStatus']}")
# print(f"Secondary status: {job['SecondaryStatus']}")

---
## What I Built
- 100K row synthetic dataset
- Local baseline training (for comparison)
- SageMaker training job with Spot instances

## Training Time Comparison

| Setup | Time | Cost |
|-------|------|------|
| Local | | |
| SageMaker ml.m5.xlarge x 1 | | |
| SageMaker ml.m5.xlarge x 2 | | |

## What I Learned
- 